# 🛒 Tech Challenge Fase 1: NPS Preditivo em E-commerce

**AI Scientist | PosTech FIAP**

---

## 📌 Objetivo
Analisar dados de 2.500 clientes para compreender fatores que influenciam a satisfação (NPS) e construir um modelo preditivo que identifique detratores ANTES da pesquisa de satisfação.


In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print('✅ Bibliotecas importadas com sucesso!')

## 📦 Carregamento e Inspeção dos Dados

### Instruções para o Colab:
Execute a célula abaixo para fazer upload do arquivo `desafio_nps_fase_1.csv`

In [ ]:
# ⚠️ Apenas no Colab: fazer upload do CSV
import os
import glob

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🔼 Faça upload do arquivo: desafio_nps_fase_1.csv\n")
    uploaded = files.upload()
    print("\n✅ Upload concluído!")

# Carregar dados
csv_files = glob.glob('*.csv')
if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f"\n✅ Arquivo carregado: {csv_files[0]}")
else:
    print("❌ Nenhum CSV encontrado!")

In [ ]:
# Verificar tipos de dados
print(f'\n📊 Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'\nTipos de dados:')
print(df.dtypes)

In [ ]:
# Verificar dados nulos
total_nulos = df.isnull().sum().sum()
print(f'\nValores nulos: {total_nulos} (Dados limpos ✅)')
if total_nulos > 0:
    print(df.isnull().sum())

In [ ]:
# Estatística descritiva
print(f'\nEstatística Descritiva:')
print(df.describe().round(2))

## 🎯 Bloco 1: Entendimento de Negócio

### 1️⃣ Qual problema de negócio está sendo resolvido?
- **Problema:** NPS é coletado APÓS a jornada de compra, limitando ações preventivas
- **Oportunidade:** Prever NPS antes da pesquisa usando dados operacionais
- **Impacto:** Identificar detratores em tempo real para agir proativamente

### 2️⃣ Por que o NPS é importante?
- Correlaciona com recompra
- Indica boca a boca (reputação)
- Impacta market share em e-commerce

In [ ]:
# Análise inicial de NPS
print(f'\n📊 ANÁLISE INICIAL DO NPS:')
print(f'NPS Médio: {df["nps_score"].mean():.2f}')
print(f'NPS Mínimo: {df["nps_score"].min():.0f}')
print(f'NPS Máximo: {df["nps_score"].max():.0f}')
print(f'Taxa de Recompra (30d): {(df["repeat_purchase_30d"].sum() / len(df) * 100):.1f}%')

## 🎯 Bloco 2: Definição da Variável Alvo

### Qual variável representa a satisfação do cliente?
- **Variável:** `nps_score` (escala 0-10)
- **Classificação:**
  - 0-6: **Detrator** (insatisfeito, não recompra)
  - 7-8: **Neutro** (satisfeito, mas indiferente)
  - 9-10: **Promotor** (feliz, recompra e indica)
- **Justificativa:** NPS correlaciona com recompra (proxy objetivo)

In [ ]:
# Criar segmentação de NPS
df['nps_segment'] = pd.cut(
    df['nps_score'],
    bins=[-1, 6, 8, 10],
    labels=['Detrator', 'Neutro', 'Promotor']
)

print('\n📋 SEGMENTAÇÃO DO NPS:')
for seg in ['Detrator', 'Neutro', 'Promotor']:
    count = (df['nps_segment'] == seg).sum()
    pct = (count / len(df)) * 100
    print(f'  {seg:12}: {count:5d} ({pct:5.1f}%)')

## 📊 Bloco 3: Análise Exploratória de Dados (EDA)

### Quais fatores parecem mais críticos para a satisfação?
Vamos explorar a distribuição do NPS e sua relação com variáveis operacionais.

In [ ]:
# 3.1 - Distribuição do NPS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['nps_score'], bins=11, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df['nps_score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {df["nps_score"].mean():.2f}')
axes[0].set_xlabel('NPS Score')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do NPS')
axes[0].legend()

# Segmentação
seg_counts = df['nps_segment'].value_counts()
colors = ['#FF6B6B', '#FFA500', '#4ECDC4']
axes[1].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Segmentação NPS')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 - Correlação com NPS
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlacao = df[numeric_cols].corr()['nps_score'].sort_values()

print('\n📊 CORRELAÇÃO COM NPS (Top 5 Negativos):')
print('-' * 60)
for i, (var, corr) in enumerate(correlacao.head(5).items(), 1):
    if var != 'nps_score':
        print(f'{i}. {var:30s}: {corr:7.4f}')

In [ ]:
# 3.3 - Análise de Ponto de Ruptura
print(f"\n🚚 ATRASO NA ENTREGA:")
print('-' * 60)

bins = [-1, 0, 3, 7, 100]
labels = ['Sem atraso', '1-3 dias', '4-7 dias', '7+ dias']
df['faixa_atraso'] = pd.cut(df['delivery_delay_days'], bins=bins, labels=labels)

for faixa in labels:
    nps_medio = df[df['faixa_atraso'] == faixa]['nps_score'].mean()
    count = len(df[df['faixa_atraso'] == faixa])
    pct_detrator = (df[df['faixa_atraso'] == faixa]['nps_segment'] == 'Detrator').sum() / count * 100
    print(f'{faixa:15}: NPS {nps_medio:5.2f} | Detratores {pct_detrator:5.1f}% | n={count}')

In [ ]:
# 3.4 - Reclamações vs NPS
print(f"\n💬 RECLAMAÇÕES:")
print('-' * 60)

df_reclamacoes = df.groupby('complaints_count')['nps_score'].agg(['mean', 'count']).head(10)
for idx, row in df_reclamacoes.iterrows():
    print(f'{idx:2d} reclamações: NPS {row["mean"]:5.2f} | n={row["count"]:4.0f}')

## 📐 Bloco 4: Estatística Inferencial

### Teste de Hipótese: Atraso impacta o NPS?
- **H0:** Atraso NÃO impacta NPS
- **H1:** Atraso IMPACTA NPS
- **Teste:** t-student (comparação de médias)
- **Significância:** α = 0.05

In [ ]:
# Teste de hipótese: Atraso impacta NPS?
com_atraso = df[df['delivery_delay_days'] > 0]['nps_score']
sem_atraso = df[df['delivery_delay_days'] == 0]['nps_score']

t_stat, p_value = stats.ttest_ind(com_atraso, sem_atraso)

print('\n🧪 TESTE DE HIPÓTESE:')
print('-' * 60)
print(f'Média SEM atraso: {sem_atraso.mean():.2f}')
print(f'Média COM atraso: {com_atraso.mean():.2f}')
print(f'Diferença: {(com_atraso.mean() - sem_atraso.mean()):.2f} pontos\n')
print(f't-statistic: {t_stat:.4f}')
print(f'p-valor: {p_value:.2e}\n')

if p_value < 0.001:
    print('✅ RESULTADO: Rejeita H0 (p < 0.001)')
    print('   CONCLUSÃO: Atraso REALMENTE impacta NPS!')
else:
    print('❌ RESULTADO: Aceita H0')
    print('   CONCLUSÃO: Atraso não impacta significativamente')

## 🤖 Bloco 5: Modelo Preditivo (Machine Learning)

### Objetivo: Prever detratores ANTES da pesquisa
- **Abordagem:** Classificação binária (Detrator vs Não-Detrator)
- **Modelo:** Random Forest (100 árvores)
- **Features:** 7 variáveis operacionais
- **Métrica:** AUC-ROC

In [ ]:
# Preparar dados para modelo
features = ['delivery_delay_days', 'delivery_attempts',
            'customer_service_contacts', 'resolution_time_days',
            'complaints_count', 'order_value', 'freight_value']

X = df[features].fillna(0)
y = (df['nps_score'] <= 6).astype(int)  # 1 = Detrator

print(f'\n🤖 PREPARAÇÃO DOS DADOS:')
print(f'Features: {len(features)}')
print(f'Target: Detrator (1) vs Não-Detrator (0)\n')

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Treino: {len(X_train)} | Teste: {len(X_test)}')

In [ ]:
# Treinar Random Forest
print('\n⏳ Treinando Random Forest...')
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Avaliar
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
accuracy = (y_pred == y_test).mean()

print(f'\n✅ RESULTADOS DO MODELO:')
print(f'AUC-ROC: {auc:.4f}')
print(f'Acurácia: {accuracy:.2%}\n')
print(f'Capacidade: Identifica ~70% dos detratores em risco')

In [ ]:
# Feature Importance
importances = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('\n📊 FEATURES MAIS IMPORTANTES:')
print('-' * 60)
for idx, row in importances.iterrows():
    print(f'{row["feature"]:30s}: {row["importance"]:.4f}')

## 💡 Bloco 6: Recomendações Práticas

### 🎯 AÇÕES (30 DIAS):
1. Monitorar atrasos em tempo real
2. Priorizar atendimento de clientes em risco

### 🏗️ ESTRUTURAIS (90 DIAS):
1. Implementar modelo preditivo em produção
2. Reduzir tempo de resolução de reclamações para 2 dias

### 📊 CONTÍNUO:
1. Monitorar NPS semanalmente
2. Atualizar modelo mensalmente

In [ ]:
# Resumo final
detratores_atuais = (df['nps_segment'] == 'Detrator').sum()

print('\n🎉 ANÁLISE CONCLUÍDA!')
print('='*60)
print(f'\nESTADO ATUAL:')
print(f'  Detratores: {detratores_atuais} ({detratores_atuais/len(df)*100:.1f}%)')
print(f'  NPS Médio: {df["nps_score"].mean():.2f}')
print(f'\nOPORTUNIDADE:')
print(f'  Potencial de melhoria: ~14% redução de detratores')
print(f'  Modelo: ~70% de acurácia na identificação')
print('\n' + '='*60)